# SampleSift: select and export OBIS occurrences

Setup once: `task frontend:setup` and `task jupyter:setup`.
Run `task dev` in one terminal and `task jupyter:lab` in another.
The notebook uses the shared OBIS backend through Vite.


In [17]:
import json
from pathlib import Path

import geopandas as gpd

from samplesift.jupyter import SampleSift

widget = SampleSift(
    api_url="http://127.0.0.1:5173/samplesift",
    collection="occurrences",
    tile_layer="occurrences",
    query_state={
        "predicates": [{"field": "depth", "operator": ">=", "value": 50}],
        "sort": ["date"],
        "selection": None,
    },
)
widget


## Draw a search region

Wait for the table to load. The initial filter selects depth of at least 50 metres.
Use the map's Polygon tool to draw a region.
Wait for the table to update before running the next cell.
The state below contains the scalar filters, sorting, and selected polygon.


In [18]:
widget.query_state


{'predicates': [{'field': 'depth', 'operator': '>=', 'value': 50}],
 'sort': ['date'],
 'selection': None}

## Alternative: select the Viking shipping forecast area

Viking covers 0–4°E and 58°30′–61°N.
Its corners are published in the [Met Office Shipping Forecast factsheet, Table 1 (PDF page 12)](https://www.metoffice.gov.uk/api/assets/file/factsheet_8_shipping_forecast_2025pdf?prefix=assets).
The included shapefile and GeoJSON were generated from those published coordinates.
They use EPSG:4326. They are not downloaded official GIS products.

Run either example below to replace your drawn polygon with Viking.
Both examples preserve the depth filter and sorting.
See `regions/README.md` for the fixture details.


In [19]:
# Load Viking from the shapefile.
viking = gpd.read_file(Path("regions/viking.shp")).to_crs("EPSG:4326")
widget.set_selection(viking.geometry.iloc[0])
viking


,area,geometry
0,Viking,"POLYGON ((4 58.5, 0 58.5, 0 61, 4 61, 4 58.5))"


## Select the same region using GeoJSON

This sets the same map polygon and search region without a geometry library.
Run this cell as an alternative to the shapefile cell above.


In [15]:
viking_geojson = json.loads(Path("regions/viking.geojson").read_text())
widget.set_selection(viking_geojson)
widget.query_state


{'predicates': [{'field': 'depth', 'operator': '>=', 'value': 50}],
 'sort': ['date'],
 'selection': {'type': 'Polygon',
  'coordinates': [[[4.0, 58.5],
    [0.0, 58.5],
    [0.0, 61.0],
    [4.0, 61.0],
    [4.0, 58.5]]]}}

Keep the shapefile's `.dbf`, `.shx`, `.prj`, and `.cpg` files beside its `.shp` file.
For your own shapefiles, reproject to EPSG:4326 before calling `set_selection()`.
The component currently supports one polygon without holes.
Check `widget.query_error` if a state cannot be applied.
Wait for the table to update before exporting.


## Link a depth slider

Release this ipywidgets slider to set the minimum depth in metres.
The observer preserves the polygon, sorting, and filters on other fields.
Wait for the table to update before exporting.


In [ ]:
import ipywidgets as widgets

depth_slider = widgets.IntSlider(value=50, min=0, max=200, step=5,
                                 description="Min depth", continuous_update=False)

def set_min_depth(change):
    predicates = [p for p in widget.query_state["predicates"] if p["field"] != "depth"]
    widget.query_state = {**widget.query_state, "predicates": predicates + [
        {"field": "depth", "operator": ">=", "value": change["new"]}
    ]}

depth_slider.observe(set_min_depth, names="value")
depth_slider


## Fetch every matching occurrence

Wait for the selected region's table to finish updating.
This fetches all pages using a fixed copy of the table's successful query.
The example uses 100 rows per request to demonstrate pagination within the smaller region.
The browser's page and map stay unchanged.
Records can change on the server during export; this is not a database snapshot.


In [16]:
df = widget.to_dataframe(all_rows=True, page_size=100)
print(f"Fetched {len(df)} matching occurrences")
df.head()


Fetched 1294 matching occurrences


,source_pk,geometry,lon,lat,depth,date,name
0,0017a578-6580-459a-abaf-b7ecc3ca6739,"{'type': 'Point', 'coordinates': ['2.7333', '5...",2.7333,58.7167,103.0,1973-01-31,Hippoglossoides platessoides
1,001044e5-5472-4ec0-bc0b-e1b9c3d126db,"{'type': 'Point', 'coordinates': ['3.4167', '6...",3.4167,60.2333,277.0,1974-01-22,Pollachius virens
2,0076dcfc-cf42-4f17-ac5a-8197b63d5518,"{'type': 'Point', 'coordinates': ['3.3333', '5...",3.3333,59.7167,245.0,1974-01-22,Melanogrammus aeglefinus
3,00cf8dbe-a099-411f-b456-5fd8991be8b6,"{'type': 'Point', 'coordinates': ['3.3333', '5...",3.3333,59.7167,245.0,1974-01-22,Argentina sphyraena
4,001eec6e-0344-432b-84a3-be59d671a8b5,"{'type': 'Point', 'coordinates': ['1.1333', '6...",1.1333,60.3667,160.0,1974-02-01,Gadus morhua


## Export the DataFrame

This writes the exact DataFrame above, without another API request.
For a direct fresh export instead, use `widget.to_csv("selected-occurrences.csv", page_size=100)`.


In [11]:
csv_path = Path("selected-occurrences.csv")
df.to_csv(csv_path, index=False)
csv_path


PosixPath('selected-occurrences.csv')